# 05 – Evaluation & Visualisierung

Dieses Notebook evaluiert das finale LightGBM-Modell und erstellt alle Auswertungsplots.

**Inhalte:**
1. Modellannahmen-Validierung (Residuenanalyse)
2. Kreuzvalidierung (TimeSeriesSplit, 3 Folds)
3. Finale Prediction-Visualisierung
4. Business-Driver-Analyse (SHAP)

In [ ]:
import sys
import numpy as np
import joblib
import polars as pl
from pathlib import Path
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, cross_validate

current_dir = Path('..') / '03_src'
sys.path.append(str(current_dir))
root_dir = Path('..')

from config import target_col, feature_cols, feature_cols_cleaned
from features import (
    add_weekdays, add_weekend, heating_season, renovation_index,
    heating_amount, add_price_features, solar_potentials,
    add_thermal_dynamics, add_cyclic_features, add_building_type_dummies
)
from utilis import (
    train_test_split, correlated_features_drop,
    validate_model_assumptions, plot_final_prediction, plot_business_drivers
)

## 5.1 Modell & Daten laden

In [ ]:
# Finales Modell laden (log-transformiert)
lgbm_model = joblib.load(str(root_dir / '05_plots' / 'lgbm_final_model_log.pkl'))

data_combined_cleaned = pl.read_parquet(
    str(root_dir / '02_data' / 'processed' / 'combined_data_cleaned.parquet')
)

df_ml = (
    data_combined_cleaned
    .pipe(add_weekdays).pipe(add_weekend).pipe(heating_season)
    .pipe(renovation_index).pipe(heating_amount).pipe(add_price_features)
    .pipe(solar_potentials).pipe(add_thermal_dynamics)
    .pipe(add_cyclic_features).pipe(add_building_type_dummies)
    .select(['date', target_col] + feature_cols)
)

X_train, X_test, y_train, y_test = train_test_split(
    df_ml, feature_cols=feature_cols, target_col=target_col
)
X_train_red = X_train[feature_cols_cleaned]
X_test_red  = X_test[feature_cols_cleaned]
X_train_final, X_test_final, final_features = correlated_features_drop(
    X_train_red, X_test_red, feautre_cols=feature_cols_cleaned
)

## 5.2 Modellannahmen validieren

In [ ]:
validate_model_assumptions(
    model=lgbm_model,
    X_test=X_test_final,
    y_test=y_test,
    output_dir=str(root_dir / '05_plots'),
    log_transform=True
)

## 5.3 Kreuzvalidierung (3 Folds)

In [ ]:
tscv = TimeSeriesSplit(n_splits=3)
scores = cross_validate(
    lgbm_model, X_train_final, y_train,
    cv=tscv,
    scoring={'MAE': 'neg_mean_absolute_error', 'R2': 'r2'},
    return_train_score=False
)

for i, (mae, r2) in enumerate(zip(-scores['test_MAE'], scores['test_R2'])):
    print(f'Fold {i+1}: MAE={mae:.3f} kWh | R²={r2:.3f}')

print(f"\nCV MAE:  {-scores['test_MAE'].mean():.3f} ± {scores['test_MAE'].std():.3f} kWh")
print(f"CV R²:   {scores['test_R2'].mean():.3f} ± {scores['test_R2'].std():.3f}")

## 5.4 Finale Prediction & Plots

In [ ]:
y_pred = np.expm1(lgbm_model.predict(X_test_final))

plot_final_prediction(
    y_test=y_test, y_pred=y_pred,
    output_dir=str(root_dir / '05_plots')
)

plot_business_drivers(
    model=lgbm_model,
    X_sample=X_test_final.sample(2000, random_state=42),
    output_dir=str(root_dir / '05_plots')
)

## 5.5 Finale Metriken

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)
print(f'MAE (Test): {mae:.3f} kWh')
print(f'R²  (Test): {r2:.3f}')